In [ ]:
import ffmpeg
import subprocess

In [ ]:
input_file = "./test/frieren_clip1.mp4"
output_file="./test/test_frieren_out.mp4"

#### NOTES

- -b adjusts bitrate, has :v at the end cuz its for video, :a for audio
- -c copies stream, it says "do not re-encode anything, just copy existing streams", so it ignores things like bitrate and anything that encodes. Good for copying exact quality and streams onto new videos

In [ ]:
cmd = [
    "ffmpeg", "-y",
    "-i", input_file,
    "-b:v", "256k",
    "bufsize", "64k", # size of the buffer that smooths spikes, not rly important but cool ig
    output_file
]
print(f"Running:", " ".join(cmd))

subprocess.run(
    cmd,
    stdout = subprocess.PIPE,
    stderr = subprocess.PIPE,
    text=True
)

#### -c:v libx264 breakdown
- -c:v = choose video codec, libx264. It's supported everywhere because it's hardware accelerated universally supported, fast, and many more reasons
- -crf = Constate Rate Factor, name is misleading tho, since it's actually a quality target that controls how aggressively frames are compressed

Why does CRF  require -c:fv? cuz CRF is encoder-specific, it has no idea how to apply it if a codec isn't specified

In [ ]:
cmd = [
    "ffmpeg", "-y",
    "-i", input_file,
    "-crf", "30",
    output_file
]

subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

In [ ]:
cmd = [
    "ffmpeg", "-y",
    "-i", input_file,
    "-ss", "3",
    "-to", "5",
    output_file
]

subprocess.run(
    cmd,
    stdout = subprocess.PIPE,
    stderr = subprocess.PIPE,
)

#### Learning openCV

In [ ]:
import cv2

cap = cv2.VideoCapture("./test/frieren_clip1.mp4")
"""Creates a video capture object:
- OpenCV opens the video file
- Prepares an internal decoder
- Gets ready to read frames sequentially
It basically connects to the video"""


ret, frame = cap.read()
"""1. OpenCV decodes the next frames
   2. Stores the pixel data in frame
   3. Sets ret to: True -> frame read successfully, False otherwise
   FRAME: The frame you're working with"""


if not ret:
    print(f"Failed to read frame")

print(type(frame))
print(frame.shape)
print(frame)

In [ ]:
"""Calculating mean for scenes and appending to txt"""
import numpy as np
import os
cap = cv2.VideoCapture("./test/frieren_clip1.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frameNum = 1
sec = 0

out = cv2.VideoWriter(
    "./test/edges.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
    isColor=True
)

with open("frame.txt", "w") as f:
    pass
with open("frame.txt", "a") as f:
    while True:
        ret, frame = cap.read()
        if frameNum == 24:
            sec += 1
            frameNum = 0
        if not ret:
            break # if we have no more frames
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # grayscale data
        edges = cv2.Canny(gray, 50, 100)               # edgedetect data

        edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

        mean_val = edges.mean()
        # print(f"FRAME #{i}: {mean_val}")
        f.write(f"{sec}:{frameNum}: {mean_val}\n")
        
        out.write(edges_bgr)
        frameNum += 1

cap.release()
out.release()

In [ ]:
cmd = [
    "ffmpeg", "-y",
    "-i", input_file,
    "-vf", "drawtext=text='Frame %{means[0]}': x=20:y=20:fontsize=24:fontcolor=white",
    output_file
]

subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

In [ ]:
import numpy as np
import math
def cosine_similarity(frame1, frame2):
    a = np.array(frame1).flatten().astype(np.float32)
    b = np.array(frame2).flatten().astype(np.float32)
    
    def magnitude(vec):
        vec = math.sqrt(
            np.sum(vec**2) 
        )
        return vec
    
    numerator = np.dot(a, b)
    denominator = magnitude(a) * magnitude(b)

    cosine = numerator / denominator

    return cosine


a = [255, 255, 255, 0, 0]
b = [0, 0, 0, 255, 150]

print(f"Cosine similarity between {a} and {b}: \n{cosine_similarity(a, b)}")

In [ ]:
import numpy as np
arr = np.array([
    [0,1,2,3,4,5,6,7,8],
    [1,2,3,4,5,6,7,8,9],
    [2,3,4,5,6,7,8,9,0],
    [3,4,5,6,7,8,9,0,1],
    [4,5,6,7,8,9,0,1,2],
    [5,6,7,8,9,0,1,2,3],
    [6,7,8,9,0,1,2,3,4],
    [7,8,9,0,1,2,3,4,5],
    [8,9,0,1,2,3,4,5,6]
])

print(f"arr len: {len(arr)}")
def pooling(arr, dim):
    # Get the # of dims for the new array 
    rows = len(arr) // dim 
    cols = len(arr[0]) // dim

    # Initializing new array
    normalized = np.zeros((rows, cols))

    # Going through entire array and skipping by dim size, ensuring subgrid doesnt overlap
    for i in range(0, len(arr), dim):
        for k in range(0, len(arr[0]), dim):
            mean = round(np.mean(arr[i:i + dim, k:k+dim]), 3) # Creating the subgrid, then capturing the mean of the entire subgrid
            normalized[i // dim, k // dim] = mean   # Placing that mean inside the corresponding new grid's index
    
    return normalized

newArr = pooling(arr, dim=3)

for row in newArr:
    print(" ".join(f"{val:4}" for val in row))

#### Plug pooling into algorithm and see what happens!


In [2]:
import numpy as np
import math
import av
import cv2

# output_file="./test/test_takopi_out.mp4"

def magnitude(vec):
    vec = math.sqrt(
        np.sum(vec**2) 
    )
    return vec

def pooling(frame, dim):
    arr = np.array(frame)

    # dividing then multiplying by dim to ensure it's divisible 
    h = (arr.shape[0] // dim) * dim
    w = (arr.shape[1] // dim) * dim
    arr = arr[:h, :w] # cropping so its perfectly divisible by dim

    # (# of blocks vertically, 'dim' rows per block, # of blocks horizontally, 'dim' rows per block)
    arr = arr.reshape(h // dim, dim, w // dim, dim)

    # print(f"Arr: {arr}")
    pooled = arr.mean(axis=(1, 3))

    return pooled

def cosine_similarity(frame1, frame2):
    a = np.array(frame1).flatten().astype(np.float32)
    b = np.array(frame2).flatten().astype(np.float32)
    
    numerator = np.dot(a, b)
    denominator = magnitude(a) * magnitude(b)

    cosine = numerator / denominator

    return cosine

def read_frames(input_file, threshold, blocksize=3):
    container = av.open(input_file)
    stream = container.streams.video[0]
    fps = float(stream.average_rate)

    prev = None
    frame_num = 0

    with open("frame.txt", "w") as f:
        for frame in container.decode(stream):
            if frame.time is None:
                continue

            timestamp_sec = float(frame.time)
            frame_num += 1

            # Reformat to 480x270 grayscale in one shot
            reformatted = frame.reformat(width=480, height=270, format="gray")
            img = reformatted.to_ndarray()

            edges = cv2.Canny(img, 50, 100)

            if np.count_nonzero(edges) == 0:
                continue

            pooled = pooling(edges, blocksize)

            if prev is not None:
                dissimilarity = abs(1 - cosine_similarity(pooled, prev))

                frame_duration = 1.0 / fps
                adjusted_sec = timestamp_sec + frame_duration
                sec = int(adjusted_sec)
                frame_in_sec = int((adjusted_sec - sec) * fps)

                f.write(f"{sec}:{frame_in_sec}: {dissimilarity:.6f}")

                if dissimilarity > threshold:
                    f.write("  | DETECTED")
                    
                if frame.key_frame:
                    f.write("  | KEYFRAME")

                f.write("\n")

            prev = pooled

    container.close()

input_file = "../test/avatar_test.mp4"
read_frames(input_file=input_file,
            threshold=0.78,
            blocksize=5)

In [ ]:
'''
==============================
    ACCURACY TEST SCRIPT
==============================
'''

import av
import cv2
import numpy as np
import math

def magnitude(vec):
    return math.sqrt(np.sum(vec**2))

def pooling(frame, dim):
    arr = np.array(frame)
    h = (arr.shape[0] // dim) * dim
    w = (arr.shape[1] // dim) * dim
    arr = arr[:h, :w]
    arr = arr.reshape(h // dim, dim, w // dim, dim)
    return arr.mean(axis=(1, 3))

def cosine_similarity(frame1, frame2):
    a = np.array(frame1).flatten().astype(np.float32)
    b = np.array(frame2).flatten().astype(np.float32)
    numerator = np.dot(a, b)
    denominator = magnitude(a) * magnitude(b)
    return numerator / denominator

def compute_dissimilarities(input_file, blocksize):
    container = av.open(input_file)
    stream = container.streams.video[0]
    fps = float(stream.average_rate)

    prev = None
    dissim_data = []  # [(sec, frame_in_sec, dissimilarity), ...]

    for frame in container.decode(stream):
        if frame.time is None:
            continue

        timestamp_sec = float(frame.time)

        frame_duration = 1.0 / fps
        adjusted = timestamp_sec + frame_duration
        sec = int(adjusted)
        frame_in_sec = int((adjusted - sec) * fps)

        reformatted = frame.reformat(width=480, height=270, format="gray")
        img = reformatted.to_ndarray()

        edges = cv2.Canny(img, 50, 100)

        if np.count_nonzero(edges) == 0:
            continue

        pooled = pooling(edges, blocksize)

        if prev is not None:
            dissimilarity = abs(1 - cosine_similarity(pooled, prev))
            dissim_data.append((sec, frame_in_sec, dissimilarity))

        prev = pooled

    container.close()
    return dissim_data

def evaluate_threshold(dissim_data, threshold, truth):
    detected = set()

    for sec, frame, dis in dissim_data:
        if dis > threshold:
            detected.add((sec, frame))

    TP = len(detected & truth)
    FP = len(detected - truth)
    FN = len(truth - detected)

    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return f1

def grid_search(input_file, truth):
    best_score = 0
    best_params = None

    for blocksize in range(3, 100):
        print(f"Testing blocksize {blocksize}")
        dissim_data = compute_dissimilarities(input_file, blocksize)

        for threshold in np.arange(0.05, 0.95, 0.01):
            score = evaluate_threshold(dissim_data, threshold, truth)

            if score > best_score:
                best_score = score
                best_params = (blocksize, threshold)

    print("Best params:", best_params)
    print("Best F1:", best_score)
    return best_params


frieren_truth = {
    (0, 13), (1, 13), (2, 13), (3, 13), (5, 13),
    (8, 1), (10, 13), (13, 13), (17, 1), (20, 13),
}

takopi_truth = {
    (5, 8), (10, 2), (11, 20), (14, 8), (18, 8),
    (27, 19), (33, 5), (35, 5), (37, 0), (40, 2), (44, 15),
}

avatar_truth = {
    (2, 20), (3, 23), (7, 8), (8, 13),
    (10, 22), (13, 1), (14, 16),
}

input_file = "../test/frieren_clip1.mp4"
grid_search(input_file=input_file, truth=frieren_truth)

Testing blocksize 3
Testing blocksize 4
Testing blocksize 5
Testing blocksize 6
Testing blocksize 7
Testing blocksize 8
Testing blocksize 9
Testing blocksize 10
Testing blocksize 11
Testing blocksize 12
Testing blocksize 13
Testing blocksize 14
Testing blocksize 15
Testing blocksize 16
Testing blocksize 17
Testing blocksize 18
Testing blocksize 19
Testing blocksize 20
Testing blocksize 21
Testing blocksize 22
Testing blocksize 23
Testing blocksize 24
Testing blocksize 25
Testing blocksize 26
Testing blocksize 27
Testing blocksize 28
Testing blocksize 29
Testing blocksize 30
Testing blocksize 31
Testing blocksize 32
Testing blocksize 33
Testing blocksize 34
Testing blocksize 35
Testing blocksize 36
Testing blocksize 37
Testing blocksize 38
Testing blocksize 39
Testing blocksize 40
Testing blocksize 41
Testing blocksize 42
Testing blocksize 43
Testing blocksize 44
Testing blocksize 45
Testing blocksize 46
Testing blocksize 47
Testing blocksize 48
Testing blocksize 49
Testing blocksize 50

(5, np.float64(0.7800000000000001))

In [ ]:
import numpy as np
# Test
def pooling(frame, dim):
    arr = np.array(frame)

    print(f"Shape before h: {arr.shape[0]} w: {arr.shape[1]}")
    # dividing then multiplying by dim to ensure it's divisible 
    h = (arr.shape[0] // dim) * dim
    w = (arr.shape[1] // dim) * dim
    arr = arr[:h, :w] # cropping so its perfectly divisible by dim
    print(f"Shape after h: {arr.shape[0]} w: {arr.shape[1]}")

    # arr[block_row, row_inside_block, block_col, col_inside_block]
    arr = arr.reshape(h // dim, dim, w // dim, dim)

    print(f"Arr: \n{arr}")
    pooled = arr.mean(axis=(1, 3))

    print("Top-left block:\n", arr[0, :, 0, :])
    print("Top-right block:\n", arr[0, :, 1, :])
    print("Bottom-left block:\n", arr[1, :, 0, :])
    print("Bottom-right block:\n", arr[1, :, 1, :])
    return pooled

test_array = [
    [1,  2,  3,   4,  5,  6],
    [7,  8,  9,  10, 11, 12],
    [13,14, 15,  16, 17, 18],

    [19,20, 21,  22, 23, 24],
    [25,26, 27,  28, 29, 30],
    [31,32, 33,  34, 35, 36],
]

pooling(test_array, 3)